# Fraud Detection - Transaction Anomaly Model

Bu notebook, dbt ile BigQuery'de olusturulan `ml_transaction_fraud_features` tablosunu pandas DataFrame'e aktarir ve label olmadan anomaly detection modeli kurar.

Kullanilan kaynak tablo:

`nova-project-498911.dbt_yasemen.ml_transaction_fraud_features`

Not: Bu model supervised fraud classification degil. `is_fraud` label'i olmadigi icin `IsolationForest` ile supheli/anormal transaction skorlamasi yapar.

In [ ]:
# Gerekli paketler eksikse bu hucreyi bir kez calistir.
# VS Code'da aktif kernel'in proje .venv'i oldugundan emin ol.
# %pip install google-cloud-bigquery google-cloud-bigquery-storage pandas scikit-learn pyarrow db-dtypes matplotlib seaborn

## 1. Import ve BigQuery ayarlari

Lokal VS Code'da BigQuery'ye baglanmak icin genelde terminalde bir kez su komutu calistirman yeterli olur:

`gcloud auth application-default login`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.cloud import bigquery
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")

PROJECT_ID = "nova-project-498911"
FEATURE_TABLE = "nova-project-498911.dbt_yasemen.ml_transaction_fraud_features"

client = bigquery.Client(project=PROJECT_ID)

## 2. Feature mart'ini pandas DataFrame'e aktar

Tablo 50M transaction civarinda oldugu icin lokal notebook'ta once orneklemle basliyoruz. `SAMPLE_MOD` degerini dusurmek daha fazla veri getirir:

- `1000`: yaklasik %0.1
- `100`: yaklasik %1
- `20`: yaklasik %5

`MAX_ROWS` lokal bellekte kontrol amacli ekstra limit uygular.

In [ ]:
SAMPLE_MOD = 100
MAX_ROWS = 200_000

query = f"""
select
    transaction_uuid,
    user_id,
    service_id,
    category,
    rating,
    market_id,
    market_name,
    region,
    country_name,
    status,
    timestamp,
    transaction_hour,
    transaction_day_of_week,
    amount_usd,
    acquisition_source,
    platform_user_agent,
    referral_source,
    gps_lat,
    gps_long,
    user_segment,
    lifecycle_segment,
    service_tier,
    is_promo_period,
    user_transaction_sequence,
    user_transaction_count_1h,
    user_transaction_count_24h,
    user_amount_usd_24h,
    user_service_transaction_count_24h,
    user_avg_amount_usd_prev_20,
    user_stddev_amount_usd_prev_20,
    amount_vs_user_avg_prev_20,
    amount_zscore_user_prev_20,
    service_avg_amount_usd_prev_100,
    amount_vs_service_avg_prev_100,
    user_distinct_services_24h,
    user_distinct_markets_24h,
    seconds_since_previous_user_transaction,
    is_late_night_transaction,
    is_unsuccessful_or_refunded
from `{FEATURE_TABLE}`
where mod(abs(farm_fingerprint(transaction_uuid)), {SAMPLE_MOD}) = 0
limit {MAX_ROWS}
"""

df = client.query(query).to_dataframe(create_bqstorage_client=True)
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

## 3. Model feature setini hazirla

ID, timestamp ve ham text kolonlarini modele direkt vermiyoruz. `platform_user_agent` gibi yuksek kardinaliteli alanlar daha sonra parse edilip browser/device feature'larina ayrilabilir.

In [ ]:
numeric_features = [
    "rating",
    "transaction_hour",
    "transaction_day_of_week",
    "amount_usd",
    "gps_lat",
    "gps_long",
    "user_transaction_sequence",
    "user_transaction_count_1h",
    "user_transaction_count_24h",
    "user_amount_usd_24h",
    "user_service_transaction_count_24h",
    "user_avg_amount_usd_prev_20",
    "user_stddev_amount_usd_prev_20",
    "amount_vs_user_avg_prev_20",
    "amount_zscore_user_prev_20",
    "service_avg_amount_usd_prev_100",
    "amount_vs_service_avg_prev_100",
    "user_distinct_services_24h",
    "user_distinct_markets_24h",
    "seconds_since_previous_user_transaction",
]

categorical_features = [
    "category",
    "market_name",
    "region",
    "country_name",
    "status",
    "acquisition_source",
    "referral_source",
    "user_segment",
    "lifecycle_segment",
    "service_tier",
    "is_promo_period",
    "is_late_night_transaction",
    "is_unsuccessful_or_refunded",
]

features = numeric_features + categorical_features
model_df = df[features].copy()

for col in categorical_features:
    model_df[col] = model_df[col].astype("string").fillna("unknown")

model_df[numeric_features] = model_df[numeric_features].replace([np.inf, -np.inf], np.nan)
model_df.shape

## 4. IsolationForest modeli

`contamination=0.01` modelin yaklasik %1 transaction'i anomaly olarak isaretlemesini bekledigimizi soyler. Gercek fraud label olmadigi icin bu deger is kurali gibi dusunulmeli ve deneme yapilarak ayarlanmalidir.

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "isolation_forest",
            IsolationForest(
                n_estimators=200,
                contamination=0.01,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

model.fit(model_df)

## 5. Skorlari uret

`IsolationForest.predict` sonucu:

- `1`: normal
- `-1`: anomaly / supheli

`fraud_anomaly_score` buyudukce transaction daha supheli kabul edilir.

In [ ]:
df["is_anomaly"] = model.predict(model_df)
df["fraud_anomaly_score"] = -model.decision_function(model_df)
df["is_suspicious_transaction"] = df["is_anomaly"].eq(-1)

df[["transaction_uuid", "fraud_anomaly_score", "is_anomaly", "is_suspicious_transaction"]].head()

In [ ]:
df["is_suspicious_transaction"].value_counts(normalize=True).rename("share")

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["fraud_anomaly_score"], bins=60)
plt.title("Fraud anomaly score distribution")
plt.xlabel("fraud_anomaly_score")
plt.show()

## 6. En supheli transaction'lari incele

In [ ]:
review_columns = [
    "transaction_uuid",
    "user_id",
    "service_id",
    "timestamp",
    "amount_usd",
    "status",
    "category",
    "market_name",
    "user_transaction_count_1h",
    "user_transaction_count_24h",
    "user_amount_usd_24h",
    "amount_zscore_user_prev_20",
    "seconds_since_previous_user_transaction",
    "fraud_anomaly_score",
    "is_suspicious_transaction",
]

top_suspicious = df.sort_values("fraud_anomaly_score", ascending=False).head(100)
top_suspicious[review_columns]

## 7. Sonuclari lokal CSV olarak kaydet

In [ ]:
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

score_columns = [
    "transaction_uuid",
    "user_id",
    "service_id",
    "timestamp",
    "fraud_anomaly_score",
    "is_anomaly",
    "is_suspicious_transaction",
]

scores_path = output_dir / "fraud_detection_scores_sample.csv"
df[score_columns].to_csv(scores_path, index=False)
scores_path

## 8. Skorlari BigQuery'ye yaz

Bu hucre notebook'un uretdigi skor sonucunu BigQuery'ye yazar. dbt tarafindaki Tableau mart'i bu tabloyu okuyup transaction detaylariyla birlestirir.

In [ ]:
destination_table = "nova-project-498911.dbt_yasemen.ml_transaction_fraud_scores_sample"
scores_df = df[score_columns].copy()

job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
load_job = client.load_table_from_dataframe(scores_df, destination_table, job_config=job_config)
load_job.result()

print(f"Wrote {len(scores_df):,} rows to {destination_table}")